# Context Managers — How `with` Actually Works

This notebook explains:
1. What a context manager is and why it exists
2. How `with` works step by step
3. The `__enter__` / `__exit__` class approach
4. The `@contextmanager` shortcut
5. How this applies to `stream_with_retry` in `chat.py`

---
## 1. The problem `with` solves

Say you open a file. You need to close it when you're done — even if an exception is thrown midway.

In [1]:
# The fragile way — if something_that_might_fail() raises, f is never closed
f = open('/tmp/test.txt', 'w')
f.write('hello')
f.close()  # never reached if the line above raises

In [ ]:
# The safe way — with guarantees cleanup no matter what
with open('/tmp/test.txt', 'w') as f:
    f.write('hello')
# f.close() is called here automatically — even if f.write() raised

print('file closed:', f.closed)

**`with` is a guarantee: whatever is inside the block, setup and teardown always happen.**

The thing being `with`-ed (here: `open(...)`) is called a **context manager**. It has two jobs:
- **set up** when you enter the `with` block
- **tear down** when you leave it (normally or via exception)

---
## 2. How `with` works — the protocol

When Python sees:
```python
with SomeThing() as x:
    do_stuff(x)
```

It does exactly this, in order:
1. Call `SomeThing().__enter__()` → the return value is bound to `x`
2. Run `do_stuff(x)`
3. Call `SomeThing().__exit__(exc_type, exc_val, exc_tb)` — always, no matter what
   - If there was no exception: all three args are `None`
   - If there was an exception: the args describe it
   - If `__exit__` returns `True`, the exception is suppressed
   - If it returns `False` (or `None`), the exception propagates normally

Let's build one from scratch to see this clearly.

In [2]:
class Loudly:
    """A context manager that just prints what's happening."""

    def __enter__(self):
        print('__enter__ called — setting up')
        return 'the value of x'   # this is what `as x` receives

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f'__exit__ called — tearing down')
        print(f'  exc_type={exc_type}, exc_val={exc_val}')
        return False  # don't suppress any exception


print('=== no exception ===')
with Loudly() as x:
    print(f'  inside block, x={x!r}')

=== no exception ===
__enter__ called — setting up
  inside block, x='the value of x'
__exit__ called — tearing down
  exc_type=None, exc_val=None


In [3]:
print('=== with exception ===')
try:
    with Loudly() as x:
        print('  about to raise')
        raise ValueError('something went wrong')
        print('  this line never runs')
except ValueError as e:
    print(f'caught outside: {e}')

=== with exception ===
__enter__ called — setting up
  about to raise
__exit__ called — tearing down
  exc_type=<class 'ValueError'>, exc_val=something went wrong
caught outside: something went wrong


Notice: **`__exit__` was still called even though an exception was raised.** That's the whole point.

The exception then propagated because `__exit__` returned `False`.

In [ ]:
class Suppressor:
    """A context manager that eats exceptions."""

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is ValueError:
            print('suppressing ValueError')
            return True   # swallow it
        return False      # let anything else propagate


with Suppressor():
    raise ValueError('this will be swallowed')

print('execution continues here because the exception was suppressed')

---
## 3. The `@contextmanager` shortcut

Writing `__enter__` and `__exit__` every time is verbose. Python's `contextlib.contextmanager` lets you write the same thing as a **generator function** — a function that uses `yield` instead of `return`.

The rule is simple:
- Everything **before** `yield` = `__enter__` (setup)
- The `yield` value = what `as x` receives
- Everything **after** `yield` = `__exit__` (teardown)
- Wrap the `yield` in `try/finally` if you need guaranteed cleanup

In [4]:
from contextlib import contextmanager

@contextmanager
def loudly():
    print('setup (was __enter__)')
    try:
        yield 'the value of x'   # caller gets this as `x` in `with loudly() as x`
    finally:
        print('teardown (was __exit__)')  # always runs


print('=== no exception ===')
with loudly() as x:
    print(f'  inside block, x={x!r}')

=== no exception ===
setup (was __enter__)
  inside block, x='the value of x'
teardown (was __exit__)


In [5]:
print('=== with exception ===')
try:
    with loudly() as x:
        raise ValueError('something went wrong')
except ValueError as e:
    print(f'caught outside: {e}')

=== with exception ===
setup (was __enter__)
teardown (was __exit__)
caught outside: something went wrong


**Same behaviour as the class, with far less code.**

The `@contextmanager` decorator handles all the `__enter__`/`__exit__` wiring for you.
The generator is paused at `yield`, the caller's block runs, then the generator resumes.

One key thing: **a `@contextmanager` function must `yield` exactly once.** It's not a loop, just pause-and-resume.

---
## 4. Putting it together — a realistic example: managed DB connection

In [ ]:
import sqlite3
from contextlib import contextmanager

@contextmanager
def db_connection(path):
    """Open a SQLite connection, yield it, always close it."""
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row
    print('connection opened')
    try:
        yield conn
    finally:
        conn.close()
        print('connection closed')


with db_connection(':memory:') as conn:
    conn.execute('CREATE TABLE t (x INTEGER)')
    conn.execute('INSERT INTO t VALUES (42)')
    row = conn.execute('SELECT x FROM t').fetchone()
    print(f'got: {row[0]}')

---
## 5. Now: why this matters for `stream_with_retry`

Here's the situation in `chat.py`.

The Anthropic SDK's streaming call is itself a context manager:

```python
with client.messages.stream(...) as stream:
    for text in stream.text_stream:
        yield f"data: {json.dumps(text)}\n\n"
    final = stream.get_final_message()
```

The problem: **`call_with_retry` calls `client.messages.create()`** — the non-streaming version. It has no way to wrap a `with` block.

So `chat_stream` was doing this:
```python
client = anthropic.Anthropic()   # creates a brand new client every call
with client.messages.stream(...) as stream:
    ...
```

Two problems:
1. **New client every call** — we already have a singleton in `anthropic_client.py` via `get_client()`, creating another is wasteful
2. **No retry** — if the API returns 429 (rate limit) or 529 (overload) at stream initiation, we just crash

The fix is `stream_with_retry`: a context manager that:
- uses `get_client()` (the singleton)
- retries the stream initiation on transient errors
- yields the `stream` object to the caller once successfully opened

In [ ]:
# Here's stream_with_retry in isolation — no Anthropic needed to understand the structure

import time
import random
from contextlib import contextmanager

MAX_RETRIES = 4
BASE_DELAY  = 2.0
MAX_DELAY   = 60.0

def _is_retryable(e):
    # in the real version this checks for RateLimitError, APIConnectionError, etc.
    return isinstance(e, IOError)

def _backoff(attempt):
    delay = min(BASE_DELAY * (2 ** attempt), MAX_DELAY)
    return delay + random.uniform(0, delay * 0.2)

def get_client():
    return 'the_singleton_client'   # placeholder


@contextmanager
def stream_with_retry(**kwargs):
    """
    Context manager. Retries stream initiation on transient errors.
    Yields the open stream object to the caller.

    Usage:
        with stream_with_retry(model=..., messages=...) as stream:
            for text in stream.text_stream:
                ...
            final = stream.get_final_message()
    """
    client = get_client()
    for attempt in range(MAX_RETRIES + 1):
        try:
            # --- SETUP: open the stream ---
            # In the real version: with client.messages.stream(**kwargs) as stream:
            # Here we simulate it:
            print(f'  attempt {attempt + 1}: opening stream...')
            stream = f'open_stream_object(attempt={attempt})'

            yield stream      # <-- caller's `with` block runs here
            return            # clean exit — we're done

        except Exception as e:
            if _is_retryable(e) and attempt < MAX_RETRIES:
                delay = _backoff(attempt)
                print(f'  retryable error: {e} — waiting {delay:.1f}s')
                time.sleep(0.01)  # shortened for demo
            else:
                raise   # non-retryable or out of attempts — propagate


# Normal usage
print('=== success on first try ===')
with stream_with_retry(model='claude-sonnet-4-6') as stream:
    print(f'  got stream: {stream}')
    print(f'  streaming tokens...')

---
## 6. One subtlety: when does retry help vs. not?

Retrying makes sense for errors that happen **at stream initiation** (before the first token) — e.g. a 429 rate limit response from the API. The stream hasn't started yet, so restarting is clean.

It does **not** help for errors that happen **mid-stream** (after tokens have already been yielded to the browser). At that point the client has already received partial output; restarting would send duplicate content. So we let mid-stream errors propagate — that's acceptable behaviour for a chat UI (the user sees a broken response and can resend).

---
## 7. Summary

| Concept | What it is |
|---|---|
| `with x as y:` | guaranteed setup + teardown around a block |
| `__enter__` | called on entry, return value = `y` |
| `__exit__` | called on exit, always, even on exception |
| `@contextmanager` | shortcut: write a generator, `yield` once, `try/finally` for cleanup |
| before `yield` | = `__enter__` (setup) |
| after `yield` / `finally` | = `__exit__` (teardown) |
| `stream_with_retry` | wraps `client.messages.stream()` with retry + singleton client, yields the stream |

The `@contextmanager` pattern lets us add retry logic **around** the SDK's own context manager, which is exactly the wrapper we need.